# ============================================================
# Étude épidémiologique : Analyse propagation virale (COVID-19)
# ============================================================

In [ ]:
# Imports des Modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import plotly.express as px
import sys


# -----------------------------
# 1. Chargement des données
# -----------------------------

In [ ]:
# URL officielle des données Google COVID-19 Open Data
url = "https://storage.googleapis.com/covid19-open-data/v3/epidemiology.csv"

# Charger directement sans enregistrer en local
usecols=["date","location_key","new_confirmed","cumulative_confirmed","new_deceased"]
df = pd.read_csv(url, usecols=usecols, parse_dates=["date"])

df.head()



In [ ]:
# Nombre de pays
print("Nombre de pays/territoires:", df["location_key"].nunique()) #?

# Dates min / max
print("Période:", df["date"].min(), "→", df["date"].max())


# -----------------------------
# 2. Exploitation et affichage des données
# -----------------------------

In [ ]:
country = "FR"   # code ISO pour la France
df_country = df[df["location_key"] == country].copy()

df_country.set_index("date")[["new_confirmed"]].plot(figsize=(12,4), title=f"Nouveaux cas journaliers - {country}")
plt.show()

df_country.set_index("date")[["cumulative_confirmed"]].plot(figsize=(12,4), title=f"Cas cumulés - {country}")
plt.show()


# -----------------------------
# 3. Interprétation des données
#    Explication des ourtils de mesure
# -----------------------------

R₀ (taux de reproduction de base)
→ C’est une constante théorique qui décrit au tout début d’une épidémie le nombre moyen de personnes infectées par un individu malade dans une population totalement susceptible (aucune immunité, aucun vaccin, aucune mesure de contrôle).
Exemple : si R₀ = 3, une personne infectée contamine en moyenne 3 autres.

Rₜ (taux de reproduction effectif, ou instantané)
→ C’est une version dynamique de R₀ : il évolue dans le temps en fonction de l’immunité acquise, des mesures sanitaires, des comportements sociaux, etc.
Exemple : si R₀ = 3 mais que 50% de la population est immunisée, alors Rₜ sera inférieur à 3.

In [ ]:
# Logarithme des cas cumulés
df_country["log_cases"] = np.log1p(df_country["cumulative_confirmed"])

# Croissance journalière
df_country["growth_rate"] = df_country["log_cases"].diff()

# Estimation simple de R_t (avec période infectieuse moyenne de 5 jours)
infectious_period = 5
df_country["R_t"] = 1 + df_country["growth_rate"] * infectious_period

df_country.plot(x="date", y="R_t", figsize=(12,4), title=f"Estimation Rₜ - {country}")
plt.axhline(1, color="red", linestyle="--")
plt.show()


# -----------------------------
# 3.1 Création du dictionnaire ISO-2 -> ISO-3
# -----------------------------

In [ ]:
# Mapping ISO-2 → ISO-3 pour tous les pays
iso2_to_iso3 = {
    "AD": "AND","AE": "ARE","AF": "AFG","AG": "ATG","AI": "AIA","AL": "ALB","AM": "ARM",
    "AO": "AGO","AR": "ARG","AT": "AUT","AU": "AUS","AW": "ABW","AX": "ALA","AZ": "AZE",
    "BA": "BIH","BB": "BRB","BD": "BGD","BE": "BEL","BF": "BFA","BG": "BGR","BH": "BHR",
    "BI": "BDI","BJ": "BEN","BL": "BLM","BM": "BMU","BN": "BRN","BO": "BOL","BQ": "BES",
    "BR": "BRA","BS": "BHS","BT": "BTN","BV": "BVT","BW": "BWA","BY": "BLR","BZ": "BLZ",
    "CA": "CAN","CC": "CCK","CD": "COD","CF": "CAF","CG": "COG","CH": "CHE","CI": "CIV",
    "CK": "COK","CL": "CHL","CM": "CMR","CN": "CHN","CO": "COL","CR": "CRI","CU": "CUB",
    "CV": "CPV","CW": "CUW","CX": "CXR","CY": "CYP","CZ": "CZE","DE": "DEU","DJ": "DJI",
    "DK": "DNK","DM": "DMA","DO": "DOM","DZ": "DZA","EC": "ECU","EE": "EST","EG": "EGY",
    "EH": "ESH","ER": "ERI","ES": "ESP","ET": "ETH","FI": "FIN","FJ": "FJI","FK": "FLK",
    "FM": "FSM","FO": "FRO","FR": "FRA","GA": "GAB","GB": "GBR","GD": "GRD","GE": "GEO",
    "GF": "GUF","GG": "GGY","GH": "GHA","GI": "GIB","GL": "GRL","GM": "GMB","GN": "GIN",
    "GP": "GLP","GQ": "GNQ","GR": "GRC","GT": "GTM","GU": "GUM","GW": "GNB","GY": "GUY",
    "HK": "HKG","HM": "HMD","HN": "HND","HR": "HRV","HT": "HTI","HU": "HUN","ID": "IDN",
    "IE": "IRL","IL": "ISR","IM": "IMN","IN": "IND","IO": "IOT","IQ": "IRQ","IR": "IRN",
    "IS": "ISL","IT": "ITA","JE": "JEY","JM": "JAM","JO": "JOR","JP": "JPN","KE": "KEN",
    "KG": "KGZ","KH": "KHM","KI": "KIR","KM": "COM","KN": "KNA","KP": "PRK","KR": "KOR",
    "KW": "KWT","KY": "CYM","KZ": "KAZ","LA": "LAO","LB": "LBN","LC": "LCA","LI": "LIE",
    "LK": "LKA","LR": "LBR","LS": "LSO","LT": "LTU","LU": "LUX","LV": "LVA","LY": "LBY",
    "MA": "MAR","MC": "MCO","MD": "MDA","ME": "MNE","MF": "MAF","MG": "MDG","MH": "MHL",
    "MK": "MKD","ML": "MLI","MM": "MMR","MN": "MNG","MO": "MAC","MP": "MNP","MQ": "MTQ",
    "MR": "MRT","MS": "MSR","MT": "MLT","MU": "MUS","MV": "MDV","MW": "MWI","MX": "MEX",
    "MY": "MYS","MZ": "MOZ","NA": "NAM","NC": "NCL","NE": "NER","NF": "NFK","NG": "NGA",
    "NI": "NIC","NL": "NLD","NO": "NOR","NP": "NPL","NR": "NRU","NU": "NIU","NZ": "NZL",
    "OM": "OMN","PA": "PAN","PE": "PER","PF": "PYF","PG": "PNG","PH": "PHL","PK": "PAK",
    "PL": "POL","PM": "SPM","PN": "PCN","PR": "PRI","PT": "PRT","PW": "PLW","PY": "PRY",
    "QA": "QAT","RE": "REU","RO": "ROU","RS": "SRB","RU": "RUS","RW": "RWA","SA": "SAU",
    "SB": "SLB","SC": "SYC","SD": "SDN","SE": "SWE","SG": "SGP","SH": "SHN","SI": "SVN",
    "SJ": "SJM","SK": "SVK","SL": "SLE","SM": "SMR","SN": "SEN","SO": "SOM","SR": "SUR",
    "SS": "SSD","ST": "STP","SV": "SLV","SX": "SXM","SY": "SYR","SZ": "SWZ","TC": "TCA",
    "TD": "TCD","TF": "ATF","TG": "TGO","TH": "THA","TJ": "TJK","TK": "TKL","TL": "TLS",
    "TM": "TKM","TN": "TUN","TO": "TON","TR": "TUR","TT": "TTO","TV": "TUV","TZ": "TZA",
    "UA": "UKR","UG": "UGA","UM": "UMI","US": "USA","UY": "URY","UZ": "UZB","VA": "VAT",
    "VC": "VCT","VE": "VEN","VG": "VGB","VI": "VIR","VN": "VNM","VU": "VUT","WF": "WLF",
    "WS": "WSM","YE": "YEM","YT": "MYT","ZA": "ZAF","ZM": "ZMB","ZW": "ZWE"
}

In [ ]:
# Convertir la date en datetime si ce n'est pas déjà fait
df["date"] = pd.to_datetime(df["date"])

# Extraire le code ISO-2 pour toutes les lignes (2 premières lettres)
df["iso2"] = df["location_key"].str[:2]

# Faire le mapping ISO-2 → ISO-3
df["iso_alpha3"] = df["iso2"].map(iso2_to_iso3)

# Supprimer les lignes non reconnues
df = df.dropna(subset=["iso_alpha3"])

# Grouper par pays ET date pour avoir le cumul au niveau pays
df_country_date = df.groupby(["iso_alpha3", "date"], as_index=False)["cumulative_confirmed"].sum()

# Comparaison pays avant et après regroupement par zone et tranformation en iso-3
#print(df_country_date["iso_alpha3"].unique())

# Pour chaque pays, récupérer la dernière valeur disponible
latest_list = []

for cnt in df_country_date["iso_alpha3"].unique():
    df_temp = df_country_date[df_country_date["iso_alpha3"] == cnt]
    idx = df_temp["date"].idxmax()  # l'index de la dernière date pour ce pays
    latest_list.append(df_temp.loc[idx])

# Concaténer toutes les dernières lignes dans un seul DataFrame
df_latest = pd.DataFrame(latest_list).reset_index(drop=True)

#print(df_latest["iso_alpha3"].unique())  # devrait contenir tous les pays


# -----------------------------
# 3.2 Création de la carte des cas cumulés
# -----------------------------

In [ ]:
# Affichage via valeur relative (% de la valeur maximale)
df_latest["cumulative_percent"] = df_latest["cumulative_confirmed"] / df_latest["cumulative_confirmed"].max() * 100

fig_pct = px.choropleth(
    df_latest,
    locations="iso_alpha3",
    color="cumulative_percent",
    hover_name="iso_alpha3",
    color_continuous_scale="Reds",
    title=f"Cas cumulés COVID-19 (% de la valeur max) au {df_latest['date'].max()}",
    projection="natural earth"
)
fig_pct.update_coloraxes(colorbar_title="% du max")
fig_pct.show()


In [ ]:
df_country.set_index("date")[["new_deceased"]].plot(figsize=(12,4), title=f"Nouveaux décès journaliers - {country}")
plt.show()


# -----------------------------
# 3.3 Comparaison des données
# -----------------------------

In [ ]:
# Comparer plusieurs pays
countries = ["FR", "IT", "ES"]
df_sel = df[df["location_key"].isin(countries)]

plt.figure(figsize=(12,5))
for c in countries:
    plt.plot(df_sel[df_sel["location_key"]==c]["date"],
             df_sel[df_sel["location_key"]==c]["cumulative_confirmed"],
             label=c)
plt.legend()
plt.title("Comparaison cas cumulés sur une période")
plt.show()

plt.figure(figsize=(12,5))
for c in countries:
    plt.plot(df_sel[df_sel["location_key"]==c]["date"],
             df_sel[df_sel["location_key"]==c]["new_deceased"],
             label=c)
plt.legend()
plt.title("Comparaison des nouvaux cas sur une période")
plt.show()


# -----------------------------
# 4 Pour aller plus loin
# -----------------------------

# -----------------------------
# 4.1 Etude des corrélations inter-variables
# -----------------------------

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests

# Exemple : on suppose que tu travailles sur les USA
country = "USA"
df_country = df_country_date[df_country_date["iso_alpha3"] == country].copy()

# Ajouter des variables de décalage (lags) sur les cas cumulés quotidiens
df_country["new_cases"] = df_country["cumulative_confirmed"].diff().fillna(0)

# Créer des lags (ici jusqu’à 14 jours)
for lag in [7, 14]:
    df_country[f"lag_{lag}"] = df_country["new_cases"].shift(lag)

# Supprimer les lignes avec NaN causées par les lags
df_country = df_country.dropna()

# On sélectionne uniquement les colonnes d’intérêt
features = ["lag_7", "lag_14"]
corrs = df_country[features + ["new_cases"]].corr()["new_cases"].drop("new_cases")
corrs = corrs.sort_values(ascending=False)

print("\nCorrélations des lags avec les nouveaux cas :\n", corrs)

# Test de causalité de Granger (exemple avec lag=14)
print("\n--- Test de Granger ---")
granger_res = grangercausalitytests(
    df_country[["new_cases", "lag_14"]], 
    maxlag=14, verbose=True
)

# Régression explicative simple : prédire new_cases avec ses lags
X = df_country[["lag_7", "lag_14"]]
y = df_country["new_cases"]
X = sm.add_constant(X)  # Ajouter constante
model = sm.OLS(y, X).fit()
print(model.summary())


Analyse du modèle de Granger et de la régression OLS
1. Test de causalité de Granger

Le test de Granger permet de déterminer si une série temporelle X "cause" une autre série Y, dans le sens où les valeurs passées de X apportent une information utile pour prédire Y.

Résultats principaux

Nombre de lags testés : de 1 à 14

Tests effectués :

F-test basé sur SSR

Chi2-test basé sur SSR

Likelihood ratio test

Parameter F-test

Interprétation des p-values
Lag	p-value (F-test)
1	0.969
2	0.9985
3	1.000
4	1.000
...	...
14	1.000

Les p-values sont toutes très élevées, proches ou égales à 1.

Règle : si p-value < 0.05 → on rejette l’hypothèse nulle (il y a causalité).
Ici, toutes les p-values > 0.05 → aucune causalité de Granger détectée pour tous les lags testés.

Conclusion Granger

Les valeurs passées de la série testée n’apportent aucune information significative pour prédire new_cases.

La série est donc non causale de Granger sur la variable cible.

2. Régression OLS (Ordinary Least Squares)

Le modèle OLS utilisé :

new_cases_𝑡 =const + 𝛽1⋅lag_7 + 𝛽2⋅lag_14 + 𝜖_𝑡⋅new_cases_t = const + β1⋅lag_7 + β2⋅lag_14 + ϵ_t​

Résultats principaux

| Variable | Coefficient | Std. Err | t     | P>|t| |  Interprétation  |

| const    | -9663.59    | 162,000  | -0.06 | 0.952 | Non significatif |

| lag_7    | 0.027       | 0.063    | 0.43  | 0.669 | Non significatif |

| lag_14   | 0.022       | 0.063    | 0.35  | 0.723 | Non significatif |

R² = 0.000 : le modèle n’explique aucune variance de new_cases.

Adj. R² = -0.002 : pénalisation pour les variables inutiles → confirme l’absence de prédictivité.

F-statistic = 0.158, p = 0.854 : le modèle global n’est pas significatif.

Durbin-Watson = 0.868 : indique une forte autocorrélation des résidus.

Omnibus et Jarque-Bera : test de normalité des résidus → très significatif → résidus très loin de la normalité.

Interprétation

Le modèle ne prédit pas les nouvelles infections à partir des lags 7 et 14 jours.

Les résidus présentent de fortes anomalies : autocorrélation et distribution fortement asymétrique et leptokurtique.

Conclusion : le modèle est inadapté pour cette série.

3. Conclusion générale

Test de Granger : aucune causalité détectée.

Régression OLS : modèle non significatif et mauvaise qualité des résidus.

Interprétation globale :

Les lags 7 et 14 jours de new_cases ne sont pas de bons prédicteurs.

La série new_cases est probablement trop volatile ou non stationnaire, ou dépend d’autres facteurs exogènes non inclus dans le modèle.

# -----------------------------
# 4.2 Algorithme de prédiction
# -----------------------------

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gamma
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

# 1) Choix et préparation des données sélectionnées (df_country : date, new_confirmed)
df_country = df[df["location_key"] == "FR"].copy()  # <-- pays au choix
df_country = df_country.sort_values("date")
y = df_country["new_confirmed"].fillna(0).clip(lower=0).astype(float)

# Lissage léger (moyenne mobile 7j)
y_smooth = y.rolling(7, min_periods=1).mean().values

# 2) Distribution d'intervalle sériel avec le Gamma discrétisée
def serial_interval_pmf(k_max=21, mean=5.0, sd=2.0):

    # Convert mean/sd -> shape, scale for Gamma
    var = sd**2
    shape = (mean**2)/var
    scale = var/mean
    
    # prob mass between k-0.5 and k+0.5 (discretisation)
    ks = np.arange(0, k_max+1)
    cdf = gamma.cdf(ks+0.5, a=shape, scale=scale) - gamma.cdf(ks-0.5, a=shape, scale=scale)
    pmf = np.maximum(cdf, 0)
    pmf = pmf / pmf.sum()
    return pmf

w = serial_interval_pmf(k_max=21, mean=5.0, sd=2.0)

# 3) Modèle de renouvellement: I_t ≈ R_t * (I * w)_t
#    On met un Kalman 1D sur x_t = log(R_t), obs: log(I_t) - log((I*w)_t)
I = y_smooth
conv = np.convolve(I, w, mode="full")[:len(I)]  # force même longueur
conv = np.where(conv<=0, 1e-6, conv)            # éviter log(0)

obs = np.log(np.where(I<=0, 1e-6, I)) - np.log(conv)

# 4) Kalman simple sur x_t = log(R_t)
T = len(obs)
x = np.zeros(T)     # état lissé (log R_t)
P = np.zeros(T)     # variance état lissé
# hyperparamètres (à ajuster)
Q = 0.02  # bruit d'état (variabilité de R_t dans le temps)
R = 0.10  # bruit d'observation

# init
x_pred = 0.0        # log(R_0) ~ 0 => R_0~1
P_pred = 1.0

for t in range(T):
    # Update (si obs[t] est nan, on saute l'update)
    if np.isfinite(obs[t]):
        Kt = P_pred / (P_pred + R)
        x_upd = x_pred + Kt * (obs[t] - x_pred)    # ici H=1
        P_upd = (1 - Kt) * P_pred
    else:
        x_upd, P_upd = x_pred, P_pred

    x[t], P[t] = x_upd, P_upd

    # Predict next
    x_pred = x_upd  # F=1
    P_pred = P_upd + Q

R_t = np.exp(x)

# 5) Prévision h jours à venir via renouvellement avec R_t_last (ou scénario)
h = 21
I_fore = I.copy()
R_last = R_t[-1]

for step in range(h):
    # Option A (baseline): R_t constant = R_last
    # Option B: R_t tend vers 1 (contrôle) => R_last = 0.5*R_last + 0.5*1
    R_last = R_last  # garde constant ici; remplace si tu veux un scénario

    # calcul de (I * w) à la volée (avec les dernières valeurs)
    conv_step = 0.0
    for k in range(len(w)):
        idx = len(I_fore)-1 - k
        if idx >= 0:
            conv_step += I_fore[idx] * w[k]
    I_next = max(R_last * conv_step, 0.0)
    I_fore = np.append(I_fore, I_next)

forecast = I_fore[-h:]

# 6) Détection de pics (observé + prévision)
peaks_obs, _ = find_peaks(I, distance=7)
peaks_fore, _ = find_peaks(forecast, distance=7)

# 7) Plots
plt.figure(figsize=(12,5))
plt.plot(df_country["date"], I, label="Incidence lissée (7j)")
plt.plot(pd.date_range(df_country["date"].iloc[-1] + pd.Timedelta(days=1), periods=h),
         forecast, label="Prévision (h=21j)", linestyle="--")
plt.scatter(df_country["date"].iloc[peaks_obs], I[peaks_obs], marker="o", label="Pics observés")
plt.scatter(pd.date_range(df_country["date"].iloc[-1] + pd.Timedelta(days=1), periods=h)[peaks_fore],
            forecast[peaks_fore], marker="x", label="Pics prévus")
plt.title("Prévision & détection de pics (modèle de renouvellement + Kalman sur R_t)")
plt.legend()
plt.show()

# 8) Courbe R_t
plt.figure(figsize=(12,4))
plt.plot(df_country["date"], R_t)
plt.axhline(1.0, ls="--", label="Seuil épidémique")
plt.title("Estimation R_t (Kalman sur log R_t)")
plt.legend()
plt.show()


# -----------------------------
# 4.2bis Et sa version ML
# -----------------------------

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Définition des tableaux de données prédictives, pour chaques modèles
pred_full_xgb = np.full(len(df_country), np.nan)
pred_full_prophet = np.full(len(df_country), np.nan)
pred_full_sarima = np.full(len(df_country), np.nan)

# === 0. Prétraitement des données
df_country = df[df["location_key"] == "FR"].copy()  # <-- pays au choix
df_country = df_country.sort_values("date")
df_country["y"] = df_country["new_confirmed"].fillna(0).clip(lower=0).astype(float)
df_country["y_log"] = np.log1p(df_country["y"])

# === 1. Création des features ===
def create_features(df, target_col="new_confirmed"):
    df_feat = pd.DataFrame(index=df.index)
    df_feat["y"] = df[target_col]
    df_feat["y_log"] = np.log1p(df_feat["y"].astype(float))

    # Lags
    for lag in [7, 14, 21]:
        df_feat[f"lag_{lag}"] = df[target_col].shift(lag)
    
    # Moyennes mobiles
    for win in [7, 14]:
        df_feat[f"ma_{win}"] = df[target_col].rolling(win).mean()
    
    # Jour de la semaine
    df_feat["dow"] = pd.to_datetime(df["date"]).dt.dayofweek
    
    return df_feat.dropna()

# Préparation df_features et df_country
df_features = create_features(df_country)
X = df_features.drop(columns=["y","y_log"])
y = df_features["y_log"].values
df_country["date"] = pd.to_datetime(df_country["date"])
df_country = df_country.set_index("date").asfreq("D")  # un jour = une ligne

# === 2. Cross-validation temporelle ===
tscv = TimeSeriesSplit(n_splits=5)
pred_xgb = np.zeros_like(y, dtype=float)

for train_idx, test_idx in tscv.split(X):
    model_xgb = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )
    model_xgb.fit(X.iloc[train_idx], y[train_idx])
    pred_xgb[test_idx] = model_xgb.predict(X.iloc[test_idx])

# MAE dans l’espace log
mae_log = mean_absolute_error(y, pred_xgb)

# === 3. Entraînement final sur toutes les données ===
model_xgb.fit(X, y)

# === 4. Prédiction future (14 jours) ===
future_days = 14
last_data = df_country.copy()

for i in range(future_days):
    new_date = last_data.index[-1] + pd.Timedelta(days=1)
    
    # Construire features pour cette nouvelle date
    row = {}
    for lag in [7, 14, 21]:
        row[f"lag_{lag}"] = last_data["new_confirmed"].iloc[-lag]
    for win in [7, 14]:
        row[f"ma_{win}"] = last_data["new_confirmed"].iloc[-win:].mean()
    row["dow"] = new_date.dayofweek
    
    row_df = pd.DataFrame([row], index=[new_date])
    y_pred_log = model_xgb.predict(row_df)[0]
    y_pred = np.expm1(y_pred_log)  # inverse de log1p
    last_data.loc[new_date, "new_confirmed"] = y_pred


# === 5. Visualisation ===

plt.figure(figsize=(10,5))
plt.plot(df_country.index, df_country["new_confirmed"], label="Historique")
plt.plot(last_data.index[-future_days:], last_data["new_confirmed"].iloc[-future_days:], 
         label="Prévision (14j)", linestyle="--")
plt.title("Prévision des cas quotidiens")
plt.xlabel("Date")
plt.ylabel("Nouveaux cas")
plt.legend()
plt.grid(True)
plt.show()


# Modèle Prophet

In [ ]:
from prophet import Prophet

df_prophet = df_country.reset_index()[["date","y"]].rename(columns={"date":"ds","y":"y"})
model_prophet = Prophet(daily_seasonality=True, yearly_seasonality=False, weekly_seasonality=True)
model_prophet.fit(df_prophet)
future = model_prophet.make_future_dataframe(periods=0)  # on prédit sur historique
forecast = model_prophet.predict(future)
pred_prophet = forecast["yhat"].values

# Modèle SARIMA

In [ ]:
sarima_order = (2,1,2)
seasonal_order = (1,1,1,7)
model_sarima = sm.tsa.statespace.SARIMAX(df_country["y"], order=sarima_order,
                                         seasonal_order=seasonal_order,
                                         enforce_stationarity=False,
                                         enforce_invertibility=False)
results_sarima = model_sarima.fit(disp=False)
pred_sarima = results_sarima.fittedvalues

# Comparaison différents modèles

In [ ]:
# --- 4. Visualisation comparative ---

# Afin de faire correspondre les dimensions de chaques prédiction avec l'historique, prétaitement nécessaire
# XGB
pred_full_xgb = np.full(len(df_country), np.nan)
common_idx = df_country.index.get_indexer(df_features.index)
pred_full_xgb[common_idx] = pred_xgb  # aligné correctement

# Prophet
pred_full_prophet = pred_prophet

# Sarima
pred_full_sarima = pred_sarima.values

# --- Calcul des métriques ---
valid_idx_xgb = ~np.isnan(pred_full_xgb)
mse_xgb = mean_squared_error(df_country["y"].iloc[valid_idx_xgb], pred_full_xgb[valid_idx_xgb])
mae_xgb = mean_absolute_error(df_country["y"].iloc[valid_idx_xgb], pred_full_xgb[valid_idx_xgb])

mse_prophet = mean_squared_error(df_country["y"], pred_full_prophet)
mae_prophet = mean_absolute_error(df_country["y"], pred_full_prophet)

mse_sarima = mean_squared_error(df_country["y"], pred_full_sarima)
mae_sarima = mean_absolute_error(df_country["y"], pred_full_sarima)

# --- Visualisation ---
plt.figure(figsize=(14,6))
plt.plot(df_country.index, df_country["y"], label="Historique", color="black")
plt.plot(df_country.index, pred_full_xgb, label=f"XGBoost (MSE={mse_xgb:.0f})", linestyle="--")
plt.plot(df_country.index, pred_full_prophet, label=f"Prophet (MSE={mse_prophet:.0f})", linestyle="-.")
plt.plot(df_country.index, pred_full_sarima, label=f"SARIMA (MSE={mse_sarima:.0f})", linestyle=":")
plt.title("Comparatif des modèles de prévision")
plt.xlabel("Date")
plt.ylabel("Nouveaux cas")
plt.legend()
plt.grid(True)
plt.show()

# --- Affichage des métriques ---
print(f"XGBoost:     MSE={mse_xgb:.1f}, MAE={mae_xgb:.1f}")
print(f"Prophet:     MSE={mse_prophet:.1f}, MAE={mae_prophet:.1f}")
print(f"SARIMA:      MSE={mse_sarima:.1f}, MAE={mae_sarima:.1f}")

# --- Comparaison des 3 modèles ---

La comparaison des 6 entités de mesures indique clairement que le modèle SARIMA est le plus performant: SARIMA se base sur une structure linéaire et saisonnière, ce qui le rend assez lisse tout en captant la majorité des fluctuations saisonnières, annuelles, etc. 

### Évaluation du modèle XGBoost via MAE (Mean Absolute Error)

L’erreur absolue moyenne (MAE) obtenue est de **33252 cas**.  

- **Interprétation brute :** en moyenne, le modèle se trompe de ~33 250 cas par jour dans ses prédictions, pour une moyenne de cas par jour qui est de 35594 cas:   
Soit une erreur relative de plus de 90 **%** , ce qui est beaucoup trop, le modèle prend en compte trop peu de facteur pour créer un résultat satisfaisant. En effet, les données du COVID étant très asymétrique (beaucoup de jours bas, peu de jour très haut), ainsi que des outliers (jour férié, administration,...), la prédiction est très sensible à ces changements brutaux à des valeurs extrêmes. L'ajout de l'information sur la moyenne glissante ainsi que la variance glissante, et l'application d'un log-transform devrai améliorer le model
- **Limite :** 
  - Ajouter des **facteurs explicatifs** (mobilité, restrictions, vaccination, météo, etc.). 
  - Tester des modèles **non-linéaires** (Random Forest, LSTM, Prophet). 
 


### Évaluation du modèle Prophet via MAE (Mean Absolute Error)

L’erreur absolue moyenne (MAE) obtenue est de **26000 cas**.  

- **Interprétation brute :** en moyenne, le modèle se trompe de ~26 000 cas par jour dans ses prédictions, pour une moyenne de cas par jour qui est de 35594 cas:   
Soit une erreur relative de plus de 60 **%** , ce qui est beaucoup trop, le modèle prend en compte trop peu de facteur pour créer un résultat satisfaisant. En effet, les données du COVID étant très asymétrique (beaucoup de jours bas, peu de jour très haut), ainsi que des outliers (jour férié, administration,...), la prédiction est très sensible à ces changements brutaux à des valeurs extrêmes. L'ajout de l'information sur la moyenne glissante ainsi que la variance glissante, et l'application d'un log-transform devrai améliorer le model
- **Limite :** 
  - Ajouter des **facteurs explicatifs** (mobilité, restrictions, vaccination, météo, etc.). 
  - Tester des modèles **non-linéaires** (Random Forest, LSTM, Prophet). 
 


### Évaluation du modèle Sarima via MAE (Mean Absolute Error)

L’erreur absolue moyenne (MAE) obtenue est de **26000 cas**.  

- **Interprétation brute :** en moyenne, le modèle se trompe de ~6 926 cas par jour dans ses prédictions, pour une moyenne de cas par jour qui est de 35594 cas:   
Soit une erreur relative de 20 **%** , ce qui est beaucoup mieux que les deux modèles précédents. De meilleurs réglages pourrait encore améliorer le modèle.
- **Limite :** 
  - Ajouter des **facteurs explicatifs** (mobilité, restrictions, vaccination, météo, etc.). 
 


# 📊 Synthèse – Étude épidémiologique

## Objectifs
- Charger et explorer les données COVID-19 (Google Open Data)
- Étudier l’évolution temporelle des cas
- Estimer la dynamique de propagation par le calcul de Rₜ
- Visualiser les résultats sur une carte mondiale

## Méthodologie
1. Chargement du dataset compressé (.csv.gz) via `pandas`
2. Sélection d’un pays (exemple : France) pour les analyses détaillées
3. Calcul du taux de croissance exponentielle à partir des cas cumulés
4. Estimation du taux de reproduction effectif **Rₜ**
   - Hypothèse : période infectieuse moyenne = 5 jours
5. Visualisation :
   - Courbes temporelles (cas journaliers, cumulés, décès)
   - Estimation de Rₜ dans le temps
   - Carte mondiale des cas cumulés à la dernière date
   - Comparaison entre pays sélectionnés sur différentes catégories: "new-deceased", "cumulative_confirmed"
6. Prédiction
   - Constructions de différents modèles (XGBoost, Prophet, Sarima)
   - Evaluation et comparaison des modèles
   - Visualisation et conclusion sur les modèles

## Résultats
- Les courbes temporelles montrent des vagues épidémiques distinctes
- L’estimation de **Rₜ** met en évidence des phases > 1 (propagation) et < 1 (contrôle)
- La carte mondiale illustre les différences régionales marquées dans la propagation

## Commentaires
- L’estimation de Rₜ est sensible aux données (sous-détection, délais de reporting…)
- R₀ est une valeur théorique initiale, tandis que notre étude porte sur **Rₜ** observé
- Les données Google sont très riches : possibilité d’étendre l’étude à la vaccination, mobilité ou politiques de santé publique, ou même de faire une étude plus poussée par zone
